In [1]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [2]:
# =============================================================================
# BLOCK 2: SYNTHESIZED FEATURE ENGINEERING (CORRECTED)
# =============================================================================
print("--- Starting Block 2: Synthesized Feature Engineering ---")
def create_synthesized_features(df_train, df_test):
    # Combine for consistent processing and reset the index
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    # Store the original id for later, as reset_index will remove it
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # --- A) Brute-Force Numerical Interactions ---
    print("Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1','grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] *all_data[NUMS[j]]
    
    # --- B) Date Features ---
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['year'] = all_data['sale_date'].dt.year
    all_data['month'] = all_data['sale_date'].dt.month
    all_data['year_diff'] = all_data['year'] - all_data['year_built']
    
    # --- C) TF-IDF Text Features ---
    print("Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning','join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),max_features=128, binary=True)
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        
        # This concat will now work because both have a simple 0-based index
        all_data = pd.concat([all_data, tfidf_df], axis=1)
    
    # --- D) Log transform some of the new interaction features ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            # Add a small constant to avoid log(0)
            all_data[c] = np.log1p(all_data[c].fillna(0))
    
    # --- E) Final Cleanup ---
    print("Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city','sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    all_data.fillna(0, inplace=True)
    
    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train','sale_price'])
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train','sale_price'])
    
    # Restore the original 'id' as the index
    X.index = train_ids
    X_test.index = test_ids
    X_test = X_test[X.columns]
    return X, X_test

# We need to re-run this from the original dataframes
X, X_test = create_synthesized_features(df_train, df_test)
print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()


--- Starting Block 2: Synthesized Feature Engineering ---
Creating brute-force numerical interaction features...
Creating TF-IDF features for text columns...
Finalizing feature set...

Synthesized FE complete. Total features: 111


10

In [3]:
# =============================================================================
# BLOCK 3: TUNE CATBOOST MEAN MODEL
# =============================================================================
import catboost as cb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import optuna

# --- 1. Prepare Data for Tuning ---
# We create a single, smaller train/validation split from the full dataset
# This makes the tuning process much faster than using K-Folds for every trial.
X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
    X, y_true, test_size=0.2, random_state=RANDOM_STATE
)

# --- 2. Define the Optuna Objective Function for CatBoost ---
def objective_catboost(trial):
    """
    This function takes an Optuna 'trial' object and does the following:
    1. Defines a search space for CatBoost's hyperparameters.
    2. Trains a CatBoost model with a set of hyperparameters suggested by the trial.
    3. Evaluates the model on the validation set.
    4. Returns the validation score (RMSE), which Optuna tries to minimize.
    """
    # Define the hyperparameter search space
    params = {
        'iterations': trial.suggest_int('iterations', 1000, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 6, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        
        # Fixed parameters
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': RANDOM_STATE,
        'verbose': 0,  # Suppress verbose output during training
        'early_stopping_rounds': 100
    }

    # Initialize and train the CatBoost model with the suggested parameters
    model = cb.CatBoostRegressor(**params)
    model.fit(
        X_train_opt, y_train_opt,
        eval_set=[(X_val_opt, y_val_opt)],
        use_best_model=True
    )

    # Make predictions on the validation set
    preds = model.predict(X_val_opt)

    # Calculate and return the Root Mean Squared Error
    rmse = np.sqrt(mean_squared_error(y_val_opt, preds))
    return rmse

# --- 3. Create and Run the Optuna Study ---
# Create a study object and specify the direction is 'minimize' (for RMSE)
study_catboost = optuna.create_study(direction='minimize')

# Start the optimization process. 
# n_trials can be increased for a more thorough search, but this is a good start.
print("--- Starting CatBoost Hyperparameter Tuning... ---")
study_catboost.optimize(objective_catboost, n_trials=N_OPTUNA_TRIALS)

# --- 4. Print the Best Results ---
print("\n--- CatBoost Tuning Complete ---")
print(f"Best trial validation RMSE: ${study_catboost.best_value:,.2f}")
print("Best hyperparameters found for CatBoost:")
for key, value in study_catboost.best_params.items():
    print(f"  '{key}': {value},")

# Store the best parameters in a dictionary for later use in the K-Fold training loop
best_params_catboost = study_catboost.best_params

[I 2025-07-16 14:33:47,700] A new study created in memory with name: no-name-07d0c2f3-29fb-414e-ae67-d7bb65c68715


--- Starting CatBoost Hyperparameter Tuning... ---


[I 2025-07-16 14:34:25,149] Trial 0 finished with value: 98248.60674926743 and parameters: {'iterations': 2329, 'learning_rate': 0.07106828663102695, 'depth': 9, 'l2_leaf_reg': 0.011456642952301135, 'subsample': 0.901161247633512, 'random_strength': 0.6138817518014036, 'bagging_temperature': 0.017313451474763708}. Best is trial 0 with value: 98248.60674926743.
[I 2025-07-16 14:34:36,809] Trial 1 finished with value: 102238.67204522768 and parameters: {'iterations': 2060, 'learning_rate': 0.045731418421428596, 'depth': 6, 'l2_leaf_reg': 0.1164878228615009, 'subsample': 0.8049753368809469, 'random_strength': 0.3233229870744167, 'bagging_temperature': 0.4358155588321845}. Best is trial 0 with value: 98248.60674926743.
[I 2025-07-16 14:35:20,524] Trial 2 finished with value: 103102.35375276385 and parameters: {'iterations': 2738, 'learning_rate': 0.011253255987517573, 'depth': 9, 'l2_leaf_reg': 0.23983723000999307, 'subsample': 0.9684487605927552, 'random_strength': 0.001601446310787545, '


--- CatBoost Tuning Complete ---
Best trial validation RMSE: $98,248.61
Best hyperparameters found for CatBoost:
  'iterations': 2329,
  'learning_rate': 0.07106828663102695,
  'depth': 9,
  'l2_leaf_reg': 0.011456642952301135,
  'subsample': 0.901161247633512,
  'random_strength': 0.6138817518014036,
  'bagging_temperature': 0.017313451474763708,


In [4]:
# =============================================================================
# BLOCK 4: K-FOLD TRAINING OF CATBOOST MEAN MODEL
# =============================================================================
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error

print("\n--- STAGE 1, PART 2: K-Fold Training of CatBoost Mean Model ---")
print("# Using the optimal hyperparameters found by Optuna.")

# --- 1. Setup K-Fold and Prediction Arrays ---
# Use StratifiedKFold on 'grade' to ensure consistent splits across models
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

# Initialize arrays to store the out-of-fold (OOF) and test set predictions
oof_catboost_preds = np.zeros(len(X))
test_catboost_preds = np.zeros(len(X_test))

# --- 2. Combine Tuned Params with Fixed Params ---
# The best_params_catboost dictionary should be in memory from the previous cell
# We add the fixed parameters needed for CatBoost training here
final_params_catboost = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': RANDOM_STATE,
    'verbose': 0,  # Suppress in-loop verbosity for cleaner output
    'early_stopping_rounds': 100,
    **best_params_catboost # Unpack the tuned hyperparameters
}

# --- 3. K-Fold Training Loop ---
for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"  Training CatBoost Mean Model - Fold {fold+1}/{N_SPLITS}...")
    
    # Split the data for the current fold
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_true.iloc[train_idx], y_true.iloc[val_idx]

    # Initialize and train the CatBoost model for this fold
    model = cb.CatBoostRegressor(**final_params_catboost)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        use_best_model=True
    )

    # Generate and store predictions
    # OOF predictions are made on the validation set for this fold
    oof_catboost_preds[val_idx] = model.predict(X_val)
    # Test predictions are averaged across all 5 fold models
    test_catboost_preds += model.predict(X_test) / N_SPLITS
    
    gc.collect()

# --- 4. Final Metrics and Comparison ---
print("\n--- CatBoost K-Fold Training Complete & Performance Metrics ---")

# Calculate the final RMSE score across all OOF predictions
final_catboost_oof_rmse = np.sqrt(mean_squared_error(y_true, oof_catboost_preds))
print(f"CatBoost Final OOF RMSE: ${final_catboost_oof_rmse:,.2f}")

# For direct comparison, here is the score from your best XGBoost model
# This value is taken from the logs of the winner_v1_301 notebook
reference_xgb_rmse = 98990.27
print(f"Reference XGBoost RMSE (from winner_v1_301): ${reference_xgb_rmse:,.2f}")

# Provide a clear conclusion
if final_catboost_oof_rmse < reference_xgb_rmse:
    print("\nCONCLUSION: SUCCESS! The tuned CatBoost model is MORE accurate than the previous XGBoost model.")
else:
    print("\nCONCLUSION: The tuned CatBoost model is LESS accurate than the previous XGBoost model.")


--- STAGE 1, PART 2: K-Fold Training of CatBoost Mean Model ---
# Using the optimal hyperparameters found by Optuna.
  Training CatBoost Mean Model - Fold 1/5...
  Training CatBoost Mean Model - Fold 2/5...
  Training CatBoost Mean Model - Fold 3/5...
  Training CatBoost Mean Model - Fold 4/5...
  Training CatBoost Mean Model - Fold 5/5...

--- CatBoost K-Fold Training Complete & Performance Metrics ---
CatBoost Final OOF RMSE: $98,246.71
Reference XGBoost RMSE (from winner_v1_301): $98,990.27

CONCLUSION: SUCCESS! The tuned CatBoost model is MORE accurate than the previous XGBoost model.


In [5]:
# =======================================================================================
#
# THE CATBOOST PATH TO A NEW BEST SCORE
#
# =======================================================================================

print("\n--- Leveraging the new, more accurate CatBoost mean predictions ---")

# ASSUMPTION: You have these variables from your CatBoost training cell.
# If not, you need to load them or re-run that cell.
# oof_catboost_preds = ... (The OOF predictions from the CatBoost K-Fold)
# test_catboost_preds = ... (The test predictions from the CatBoost K-Fold)

# 1. Calculate the NEW error target. This is the absolute error of the CatBoost model.
error_target_catboost = np.abs(y_true - oof_catboost_preds)
print(f"New error target created based on CatBoost model's performance.")

# 2. Create the feature set for the error model.
# The error model's job is to predict the error based on the original features
# PLUS the mean prediction from the first stage model.
X_for_error_cb = X.copy()
X_for_error_cb['mean_pred_oof'] = oof_catboost_preds

X_test_for_error_cb = X_test.copy()
X_test_for_error_cb['mean_pred_oof'] = test_catboost_preds

print("Feature set for the error model is ready.")


--- Leveraging the new, more accurate CatBoost mean predictions ---
New error target created based on CatBoost model's performance.
Feature set for the error model is ready.


In [8]:
# --- STAGE 2 (CatBoost Path): K-Fold Training of the Error Model ---
print("\n--- STAGE 2 (CatBoost Path): Training XGBoost to predict CatBoost's errors ---")

# Use the same pre-tuned XGBoost parameters for the error model from your winner_v1_301 notebook
# best_params_error = {...} 
final_params_error = {'objective': 'reg:squarederror', 'eval_metric': 'rmse',
                      'tree_method': 'hist', 'random_state': RANDOM_STATE, 'n_jobs': -1,
                      **best_params_error}

# Initialize arrays for error predictions
oof_error_preds_cb = np.zeros(len(X))
test_error_preds_cb = np.zeros(len(X_test))

# Use the same K-Fold splits
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error_cb, grade_for_stratify)):
    print(f"  Training Error Model on CatBoost Errors - Fold {fold+1}/{N_SPLITS}...")
    
    # Define model
    model = xgb.XGBRegressor(**final_params_error, n_estimators=2000, early_stopping_rounds=100)
    
    # Fit model on the new CatBoost error target
    model.fit(X_for_error_cb.iloc[train_idx], error_target_catboost.iloc[train_idx],
              eval_set=[(X_for_error_cb.iloc[val_idx], error_target_catboost.iloc[val_idx])],
              verbose=False)
    
    # Store predictions
    oof_error_preds_cb[val_idx] = model.predict(X_for_error_cb.iloc[val_idx])
    test_error_preds_cb += model.predict(X_test_for_error_cb) / N_SPLITS

# --- Performance of the New Error Model ---
final_error_rmse_cb = np.sqrt(mean_squared_error(error_target_catboost, oof_error_preds_cb))
print("\n--- Error Model Performance ---")
print(f"Original Error Model RMSE (on XGBoost errors): $62,782.99")
print(f"New Error Model RMSE (on CatBoost errors)    : ${final_error_rmse_cb:,.2f}")


--- STAGE 2 (CatBoost Path): Training XGBoost to predict CatBoost's errors ---


NameError: name 'best_params_error' is not defined